In [ ]:

!pip -q install python-docx pymupdf pytesseract pillow
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr

print("Installation complete!")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Installation complete!


In [ ]:

import os
import re
import difflib
import html
import io

import fitz
import pytesseract

from PIL import Image
from docx import Document
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl
from docx.text.paragraph import Paragraph
from docx.table import Table

from google.colab import files
from IPython.display import display, HTML

print("Libraries imported successfully!")


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Libraries imported successfully!


In [ ]:

def normalize_text(text):
    """Normalize whitespace for fair text comparison."""
    return re.sub(r"\s+", " ", text).strip()


def calculate_similarity(original, updated):
    """Return similarity percentage between two texts."""
    a = normalize_text(original)
    b = normalize_text(updated)

    if not a and not b:
        return 100.0

    return round(difflib.SequenceMatcher(None, a, b).ratio() * 100, 2)


def compare_text(original, updated):
    """Create word-level differences and change statistics."""
    old_words = original.split()
    new_words = updated.split()

    matcher = difflib.SequenceMatcher(
        None, old_words, new_words, autojunk=False
    )

    old_html = []
    new_html = []
    added = []
    deleted = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        old_part = old_words[i1:i2]
        new_part = new_words[j1:j2]

        if tag == "equal":
            old_html.extend(html.escape(w) for w in old_part)
            new_html.extend(html.escape(w) for w in new_part)

        elif tag == "delete":
            deleted.extend(old_part)
            old_html.append(
                '<span class="deleted">' +
                html.escape(" ".join(old_part)) +
                '</span>'
            )

        elif tag == "insert":
            added.extend(new_part)
            new_html.append(
                '<span class="added">' +
                html.escape(" ".join(new_part)) +
                '</span>'
            )

        elif tag == "replace":
            deleted.extend(old_part)
            added.extend(new_part)

            old_html.append(
                '<span class="deleted">' +
                html.escape(" ".join(old_part)) +
                '</span>'
            )
            new_html.append(
                '<span class="added">' +
                html.escape(" ".join(new_part)) +
                '</span>'
            )

    return {
        "original_html": " ".join(old_html),
        "updated_html": " ".join(new_html),
        "added_words": added,
        "deleted_words": deleted,
        "similarity": calculate_similarity(original, updated),
    }


print("Comparison functions ready!")


Comparison functions ready!


In [ ]:

def run_to_html(run):
    text = html.escape(run.text).replace("\n", "<br>")

    if not text:
        return ""

    if run.bold:
        text = f"<strong>{text}</strong>"
    if run.italic:
        text = f"<em>{text}</em>"
    if run.underline:
        text = f"<u>{text}</u>"

    return text


def paragraph_to_html(paragraph):
    content = "".join(run_to_html(run) for run in paragraph.runs)

    style_name = paragraph.style.name if paragraph.style else ""
    safe_style = html.escape(style_name)

    if style_name.startswith("Heading"):
        level_match = re.search(r"(\d+)", style_name)
        level = int(level_match.group(1)) if level_match else 2
        level = max(1, min(level, 6))
        return f"<h{level}>{content}</h{level}>"

    if style_name == "Title":
        return f"<h1>{content}</h1>"

    return f"<p>{content}</p>"


def read_word_document(path):
    """Extract Word paragraphs and tables in their document order."""
    document = Document(path)
    items = []

    for element in document.element.body.iterchildren():
        if isinstance(element, CT_P):
            paragraph = Paragraph(element, document)
            if paragraph.text.strip():
                items.append({
                    "type": "paragraph",
                    "text": paragraph.text,
                    "html": paragraph_to_html(paragraph),
                })

        elif isinstance(element, CT_Tbl):
            table = Table(element, document)
            rows_text = []
            rows_html = []

            for row in table.rows:
                cells_text = [
                    normalize_text(cell.text) for cell in row.cells
                ]
                rows_text.append(" | ".join(cells_text))

                cells_html = "".join(
                    "<td>" + html.escape(cell.text).replace("\n", "<br>")
                    + "</td>"
                    for cell in row.cells
                )
                rows_html.append("<tr>" + cells_html + "</tr>")

            if rows_text:
                items.append({
                    "type": "table",
                    "text": "\n".join(rows_text),
                    "html": (
                        '<table class="doc-table"><tbody>'
                        + "".join(rows_html)
                        + "</tbody></table>"
                    ),
                })

    return items


print("Word reader ready!")


Word reader ready!


In [ ]:

def read_pdf_document(path, dpi=200):
    """Extract PDF text and use OCR on scanned or nearly empty pages."""
    document = fitz.open(path)
    items = []

    try:
        for page_number, page in enumerate(document, start=1):
            text = page.get_text("text").strip()

            # OCR scanned pages or pages with very little extractable text.
            if len(text) < 20:
                pix = page.get_pixmap(
                    matrix=fitz.Matrix(dpi / 72, dpi / 72),
                    alpha=False
                )
                image = Image.open(
                    io.BytesIO(pix.tobytes("png"))
                )
                text = pytesseract.image_to_string(image).strip()

            items.append({
                "type": "page",
                "text": text,
                "html": (
                    f"<h3>Page {page_number}</h3><p>"
                    + html.escape(text).replace("\n", "<br>")
                    + "</p>"
                ),
            })
    finally:
        document.close()

    if not any(item["text"].strip() for item in items):
        raise ValueError(
            "No text was extracted. Check the PDF or try a higher OCR DPI."
        )

    return items


print("PDF and OCR reader ready!")


PDF and OCR reader ready!


In [ ]:

# CELL 6: Display an improved side-by-side document comparison

def show_comparison(original_items, updated_items, title):
    original_text = "\n".join(x["text"] for x in original_items)
    updated_text = "\n".join(x["text"] for x in updated_items)

    result = compare_text(original_text, updated_text)

    css = """
    <style>
    .cdc {
        font-family: Arial, sans-serif;
        color: #202124;
        margin: 12px 0;
    }
    .cdc-title {
        font-size: 27px;
        font-weight: 800;
        margin: 10px 0 20px;
        color: #f8fafc;
    }
    .cdc-stats {
        display: grid;
        grid-template-columns: repeat(3, minmax(0, 1fr));
        gap: 12px;
        margin-bottom: 20px;
    }
    .cdc-stat {
        background: #f8fafc;
        border: 1px solid #dbe2ea;
        border-radius: 12px;
        padding: 14px;
        color: #334155;
    }
    .cdc-number {
        font-size: 25px;
        font-weight: 800;
        display: block;
        margin-top: 4px;
        color: #1e293b;
    }
    .cdc-panels {
        display: grid;
        grid-template-columns: minmax(0,1fr) minmax(0,1fr);
        gap: 16px;
        align-items: start;
    }
    .cdc-panel {
        background: #ffffff;
        border: 1px solid #d5dce5;
        border-radius: 12px;
        padding: 18px;
        color: #202124;
        min-width: 0;
        overflow-wrap: anywhere;
        line-height: 1.75;
        font-size: 15px;
    }
    .cdc-panel-title {
        font-size: 18px;
        font-weight: 800;
        margin: 0 0 16px;
        padding: 0 0 12px;
        border-bottom: 2px solid #e5e7eb;
    }
    .cdc-original {
        color: #b42318;
    }
    .cdc-updated {
        color: #137333;
    }
    .cdc-panel h1,
    .cdc-panel h2,
    .cdc-panel h3,
    .cdc-panel h4,
    .cdc-panel h5,
    .cdc-panel h6 {
        font-weight: 800;
        line-height: 1.35;
        margin: 16px 0 8px;
        color: #202124;
    }
    .cdc-panel p {
        margin: 8px 0;
    }
    .cdc-deleted {
        background: #ffe1e1;
        color: #9b1c1c;
        text-decoration: line-through;
        text-decoration-thickness: 1px;
        border-radius: 3px;
        padding: 1px 2px;
    }
    .cdc-added {
        background: #d4f8dc;
        color: #12632f;
        font-weight: 600;
        border-radius: 3px;
        padding: 1px 2px;
    }
    .cdc-table {
        border-collapse: collapse;
        width: 100%;
        margin: 10px 0;
    }
    .cdc-table td {
        border: 1px solid #d1d5db;
        padding: 7px;
    }
    @media (max-width: 700px) {
        .cdc-panels {
            grid-template-columns: minmax(0,1fr);
        }
        .cdc-stats {
            grid-template-columns: 1fr;
        }
    }
    </style>
    """

    def highlighted_words(old_text, new_text):
        """Highlight only changed words between two text items."""
        old_words = old_text.split()
        new_words = new_text.split()

        matcher = difflib.SequenceMatcher(
            None, old_words, new_words, autojunk=False
        )

        left, right = [], []

        for tag, a, b, c, d in matcher.get_opcodes():
            old_part = html.escape(" ".join(old_words[a:b]))
            new_part = html.escape(" ".join(new_words[c:d]))

            if tag == "equal":
                left.append(old_part)
                right.append(new_part)

            elif tag == "delete":
                left.append(
                    f'<span class="cdc-deleted">{old_part}</span>'
                )

            elif tag == "insert":
                right.append(
                    f'<span class="cdc-added">{new_part}</span>'
                )

            elif tag == "replace":
                left.append(
                    f'<span class="cdc-deleted">{old_part}</span>'
                )
                right.append(
                    f'<span class="cdc-added">{new_part}</span>'
                )

        return " ".join(left), " ".join(right)

    def make_item(text, item):
        """Keep heading levels in the rendered document."""
        if item["type"] == "paragraph":
            match = re.match(r"\s*<h([1-6])", item["html"])
            tag = "h" + match.group(1) if match else "p"
        else:
            tag = "p"

        return f"<{tag}>{text}</{tag}>"

    old_display = []
    new_display = []

    matcher = difflib.SequenceMatcher(
        None,
        [normalize_text(x["text"]) for x in original_items],
        [normalize_text(x["text"]) for x in updated_items],
        autojunk=False
    )

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        old_group = original_items[i1:i2]
        new_group = updated_items[j1:j2]

        if tag == "equal":
            for old, new in zip(old_group, new_group):
                # Preserve original formatting for unchanged content.
                old_display.append(old["html"])
                new_display.append(new["html"])

        elif tag == "delete":
            for old in old_group:
                old_display.append(
                    make_item(
                        '<span class="cdc-deleted">'
                        + html.escape(old["text"])
                        + '</span>',
                        old
                    )
                )

        elif tag == "insert":
            for new in new_group:
                new_display.append(
                    make_item(
                        '<span class="cdc-added">'
                        + html.escape(new["text"])
                        + '</span>',
                        new
                    )
                )

        elif tag == "replace":
            paired = min(len(old_group), len(new_group))

            for k in range(paired):
                old = old_group[k]
                new = new_group[k]

                left, right = highlighted_words(
                    old["text"], new["text"]
                )

                old_display.append(make_item(left, old))
                new_display.append(make_item(right, new))

            # Display unmatched extra items as complete additions/deletions.
            for old in old_group[paired:]:
                old_display.append(
                    make_item(
                        '<span class="cdc-deleted">'
                        + html.escape(old["text"])
                        + '</span>',
                        old
                    )
                )

            for new in new_group[paired:]:
                new_display.append(
                    make_item(
                        '<span class="cdc-added">'
                        + html.escape(new["text"])
                        + '</span>',
                        new
                    )
                )

    stats = f"""
    <div class="cdc-stats">
        <div class="cdc-stat">
            Similarity
            <span class="cdc-number">{result['similarity']}%</span>
        </div>
        <div class="cdc-stat">
            Added words
            <span class="cdc-number">{len(result['added_words'])}</span>
        </div>
        <div class="cdc-stat">
            Deleted words
            <span class="cdc-number">{len(result['deleted_words'])}</span>
        </div>
    </div>
    """

    panels = f"""
    <div class="cdc-panels">
        <section class="cdc-panel">
            <div class="cdc-panel-title cdc-original">
                ORIGINAL DOCUMENT
            </div>
            {''.join(old_display) or '<p>Empty document</p>'}
        </section>
        <section class="cdc-panel">
            <div class="cdc-panel-title cdc-updated">
                UPDATED DOCUMENT
            </div>
            {''.join(new_display) or '<p>Empty document</p>'}
        </section>
    </div>
    """

    display(HTML(
        css
        + '<div class="cdc">'
        + f'<div class="cdc-title">{html.escape(title)}</div>'
        + stats
        + panels
        + '</div>'
    ))

    return result


print("Document comparison display is ready!")


Document comparison display is ready!


In [ ]:

print("Upload the ORIGINAL Word document (.docx)")
original_upload = files.upload()

if not original_upload:
    raise ValueError("No original Word document uploaded.")

original_path = next(iter(original_upload))

if not original_path.lower().endswith(".docx"):
    raise ValueError("Please upload a .docx Word document.")

print("Now upload the UPDATED Word document (.docx)")
updated_upload = files.upload()

if not updated_upload:
    raise ValueError("No updated Word document uploaded.")

updated_path = next(iter(updated_upload))

if not updated_path.lower().endswith(".docx"):
    raise ValueError("Please upload a .docx Word document.")

original_word_items = read_word_document(original_path)
updated_word_items = read_word_document(updated_path)

word_result = show_comparison(
    original_word_items,
    updated_word_items,
    "Word Document Comparison"
)


Upload the ORIGINAL Word document (.docx)


Saving Community_Centre_Original_v2.docx to Community_Centre_Original_v2.docx
Now upload the UPDATED Word document (.docx)


Saving Community_Centre_Updated_v2.docx to Community_Centre_Updated_v2.docx


In [ ]:

print("Upload the ORIGINAL PDF document")
original_pdf_upload = files.upload()

if not original_pdf_upload:
    raise ValueError("No original PDF uploaded.")

original_pdf_path = next(iter(original_pdf_upload))

if not original_pdf_path.lower().endswith(".pdf"):
    raise ValueError("Please upload a .pdf file.")

print("Now upload the UPDATED PDF document")
updated_pdf_upload = files.upload()

if not updated_pdf_upload:
    raise ValueError("No updated PDF uploaded.")

updated_pdf_path = next(iter(updated_pdf_upload))

if not updated_pdf_path.lower().endswith(".pdf"):
    raise ValueError("Please upload a .pdf file.")

print("Reading original PDF...")
original_pdf_items = read_pdf_document(original_pdf_path)

print("Reading updated PDF...")
updated_pdf_items = read_pdf_document(updated_pdf_path)

pdf_result = show_comparison(
    original_pdf_items,
    updated_pdf_items,
    "PDF Document Comparison"
)


Upload the ORIGINAL PDF document


Saving College_Trip_Original.pdf to College_Trip_Original.pdf
Now upload the UPDATED PDF document


Saving College_Trip_Updated.pdf to College_Trip_Updated.pdf
Reading original PDF...
Reading updated PDF...


In [ ]:

# Change this to "word_result" or "pdf_result"
result_to_summarize = "word_result"

if result_to_summarize == "word_result":
    result = word_result
elif result_to_summarize == "pdf_result":
    result = pdf_result
else:
    raise ValueError("Choose 'word_result' or 'pdf_result'.")

print("SIMILARITY:", result["similarity"], "%")
print("ADDED WORDS:", len(result["added_words"]))
print("DELETED WORDS:", len(result["deleted_words"]))

print("\nAdded text:")
print(" ".join(result["added_words"]) or "No added words")

print("\nDeleted text:")
print(" ".join(result["deleted_words"]) or "No deleted words")


SIMILARITY: 25.82 %
ADDED WORDS: 32
DELETED WORDS: 13

Added text:
Updated digital resources, materials. 8:30 6:00 five twenty-one Wi-Fi, room, and computer access. Thursday 4:30 Monthly storytelling sessions are organised for school students. card or another valid student ID. ₹400. digital skills,

Deleted text:
Original resources. 9:00 5:00 three fourteen Wi-Fi and room. Friday 4:00 card. ₹300.
